<a href="https://colab.research.google.com/github/DineshSBhauryal/streamlit-app/blob/main/Session_10_RAG_Part_1_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialisation

In [ ]:
import nltk
from nltk import word_tokenize
from nltk import sent_tokenize

from nltk import pos_tag
from nltk.tag.mapping import map_tag

import numpy as np

nltk.download('all', quiet=True)

True

In [ ]:
def get_upos_tags(text: str):
    """
    Tokenizes the text and returns (word, UPOS) pairs.
    Example UPOS tags: NOUN, VERB, ADJ, DET, ADP, etc.
    """
    word_list = word_tokenize(text)
    treebank_tags = pos_tag(word_list)
    upos_tags = [(word, map_tag('en-ptb', 'universal', tag)) for word, tag in treebank_tags]
    return upos_tags

In [ ]:
# Install the Sentence Transformer library
!pip install --quiet --upgrade sentence-transformers

import os
import logging

# Suppress Hugging Face / Transformers logs
os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # only show errors

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer, util

# For link of all SBERT models : https://sbert.net
# Load model silently
# This model will convert the input text into an embedding
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def get_embedding(text):
    return model.encode(text,convert_to_tensor=True)

def find_text_similarity(text1, text2):
    return round(float(util.pytorch_cos_sim(get_embedding(text1), get_embedding(text2))[0][0]), 4)

def find_embedding_similarity(emb1, emb2):
    return round(float(util.pytorch_cos_sim(emb1, emb2)[0][0]), 4)

In [ ]:
def create_embedding_list(text_list):
    return [get_embedding(text) for text in text_list]

In [ ]:
def find_answer(query, text_list, embedding_list):
    # convert the query into an embedding
    query_embedding = get_embedding(query)

    # find the cosine similarity between the query embedding and all the DB text embeddings
    sim_list = [find_embedding_similarity(query_embedding, emb) for emb in embedding_list]
    print("Similarity values between query and all sentences in the list:")
    print(sim_list)
    print("")

    # Return the text whose embedding has the highest similarity with the query embedding
    return text_list[sim_list.index(max(sim_list))]

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_wikipedia_html(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/91.0.4472.114 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text

def extract_paragraphs(html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    # Main content of Wikipedia pages is inside <div id="mw-content-text">
    content_div = soup.find("div", {"id": "mw-content-text"})
    paragraphs = []

    if content_div:
        for p in content_div.find_all("p"):
            text = p.get_text().strip()
            if text:  # ignore empty or citation-only paragraphs
                paragraphs.append(text)
    return paragraphs

In [ ]:
def find_answer_top_k(query, text_list, embedding_list, k=2):
    query_embedding = get_embedding(query)
    sim_list = [find_embedding_similarity(query_embedding, emb) for emb in embedding_list]

    # Get indices of top-k similarities (sorted descending)
    top_k_indices = np.argsort(sim_list)[::-1][:k]

    # Return top-k texts as a list
    return [text_list[i] for i in top_k_indices]


In [ ]:
from openai import OpenAI

from dotenv import load_dotenv
import os
# load_dotenv("api_key.env")
# api_key = os.getenv('api_key')

# 1. Visit : https://openrouter.ai/
# 2. Login with your Gmail ID and create a new API Key (top right menu)
# 3. Create api_key.env file in the same folder as your code
# 4. Put the api_key there. Just write one line (use your own API Key, the one below wont work):
# api_key=sk-or-v1-aasdewr34asdc0r31oweijf

api_key = ""

def get_ai_response(prompt):
  client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
  )

  completion = client.chat.completions.create(
      model="openai/gpt-oss-20b:free",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ]
  )

  return completion.choices[0].message.content


# NLP Tools

In [ ]:
text = "I am learning Large Language Models. I will then build a cool application using it."

In [ ]:
# Sentence Tokenization
# Tokenization is basically breaking a piece of text
# into smaller units.
sent_tokenize(text)

['I am learning Large Language Models.',
 'I will then build a cool application using it.']

In [ ]:
# Word Tokenization
word_tokenize(text)

# What AI does is sub-word tokenization.

['I',
 'am',
 'learning',
 'Large',
 'Language',
 'Models',
 '.',
 'I',
 'will',
 'then',
 'build',
 'a',
 'cool',
 'application',
 'using',
 'it',
 '.']

In [ ]:
# POS Tagging (Parts of Speech)
print(get_upos_tags(text))

# One application is sentiment analysis.

# Named Entity Recognition [NER]
# classify words into categories..
# Company Name, Person Name, Brand, Location

[('I', 'PRON'), ('am', 'VERB'), ('learning', 'VERB'), ('Large', 'NOUN'), ('Language', 'NOUN'), ('Models', 'NOUN'), ('.', '.'), ('I', 'PRON'), ('will', 'VERB'), ('then', 'ADV'), ('build', 'VERB'), ('a', 'DET'), ('cool', 'ADJ'), ('application', 'NOUN'), ('using', 'VERB'), ('it', 'PRON'), ('.', '.')]


# LLM Tools

In [ ]:
text = "I am a human being."
get_embedding(text)

tensor([ 5.2786e-03,  1.9185e-02,  6.3271e-02,  1.1727e-02, -5.7958e-02,
        -7.7475e-02,  1.4263e-01, -3.2002e-02,  4.6559e-02,  9.9696e-03,
        -9.2031e-03, -9.0321e-02, -3.2316e-02, -1.8097e-02,  7.7099e-02,
        -3.1339e-02, -5.2647e-03, -4.3888e-02, -4.0001e-02,  4.4545e-02,
        -2.9328e-02,  4.6565e-02, -1.5779e-02, -2.0519e-02, -4.9357e-02,
        -1.5548e-02, -1.0629e-02, -3.4567e-02,  6.5551e-02, -1.7427e-02,
         1.2922e-02, -6.9742e-02,  9.9495e-02, -1.9190e-02, -3.6043e-02,
         7.4612e-02,  4.4963e-02, -5.4321e-02,  5.6025e-02,  1.8090e-02,
         3.3005e-02, -8.2020e-02, -6.1142e-03,  5.9380e-03,  3.7282e-02,
         5.0122e-02, -2.1457e-02, -4.2981e-02, -4.2478e-02, -5.2496e-02,
        -7.1995e-02,  1.0558e-01, -3.2934e-02,  6.6258e-02, -8.5375e-03,
        -4.0289e-02,  4.9874e-02,  2.4564e-02,  2.1757e-02, -5.7993e-02,
         9.0196e-03,  9.0460e-03, -1.8721e-02,  1.0411e-01,  4.4548e-02,
        -8.3939e-03,  3.7495e-02, -4.1318e-02, -1.7

In [ ]:
text1 = "I am a human being."
text2 = "What are you doing?"
find_text_similarity(text1, text2)
# So this value ranges between -1 to +1.

# ONLY TEXT DOES NOT HAVE A NATURAL NUMERIC REPRESENTATION.

0.3407

# RAG : Retrieval Augmented Generation

In [ ]:
query = "whats special about Amazon?"

In [ ]:
get_ai_response(query)

"**Amazon is special because it’s a “platform‑ecosystem” that redefines every part of the modern economy.** Below are the key elements that make it stand out:\n\n1. **Customer‑centric culture**  \n   *“We are the world’s most customer‑obsessed company.”* Pick any customer pain point and you'll see Amazon has built a solution: endless choice, low prices, instant delivery, proactive customer service.\n\n2. **Unmatched scale of logistics & speed**  \n   * **Fulfilment network** – 175+ fulfillment centers, 60+ sortation hubs, and 1,200+ delivery vans.  \n   * **Prime** – 5‑day or same‑day delivery with a growing **Amazon Flex** crowd‑source fleet.  \n   * **Amazon Air** – 13‑year‑old cargo airline that turns “next‑day” into “today” for millions of items.\n\n3. **Marketplace & ecosystem lock‑in**  \n   * **Marketplace vendors** get access to half a billion active shoppers.  \n   * **Prime members** have a “loyalty” effect that keeps them buying on Amazon longer than competitors.  \n   * **S

In [ ]:
text = """
The Amazon rainforest is the largest tropical rainforest in the world,
spanning across nine countries in South America.
It is home to an incredible diversity of wildlife,
including jaguars, toucans, and countless species of insects.
The rainforest also plays a crucial role in regulating the Earth's climate
by absorbing large amounts of carbon dioxide.
Unfortunately, deforestation and illegal logging have been
threatening this vital ecosystem for decades.
Conservation efforts are underway, but sustainable practices need to be
adopted globally to preserve this natural treasure for future generations."""

In [ ]:
text_list = sent_tokenize(text)

In [ ]:
# Each item in this list is called a "chunk"
print(text_list)

['\nThe Amazon rainforest is the largest tropical rainforest in the world,\nspanning across nine countries in South America.', 'It is home to an incredible diversity of wildlife,\nincluding jaguars, toucans, and countless species of insects.', "The rainforest also plays a crucial role in regulating the Earth's climate\nby absorbing large amounts of carbon dioxide.", 'Unfortunately, deforestation and illegal logging have been\nthreatening this vital ecosystem for decades.', 'Conservation efforts are underway, but sustainable practices need to be\nadopted globally to preserve this natural treasure for future generations.']


In [ ]:
for item in text_list:
    print(item)
    print("")


The Amazon rainforest is the largest tropical rainforest in the world,
spanning across nine countries in South America.

It is home to an incredible diversity of wildlife,
including jaguars, toucans, and countless species of insects.

The rainforest also plays a crucial role in regulating the Earth's climate
by absorbing large amounts of carbon dioxide.

Unfortunately, deforestation and illegal logging have been
threatening this vital ecosystem for decades.

Conservation efforts are underway, but sustainable practices need to be
adopted globally to preserve this natural treasure for future generations.



In [ ]:
emb_list = create_embedding_list(text_list)

In [ ]:
find_answer(query, text_list, emb_list)

Similarity values between query and all sentences in the list:
[0.3382, 0.2559, 0.1526, 0.0836, 0.0893]



'\nThe Amazon rainforest is the largest tropical rainforest in the world,\nspanning across nine countries in South America.'

In [ ]:
query = "What is needed to preserve nature?"
find_answer(query, text_list, emb_list)

Similarity values between query and all sentences in the list:
[0.0759, 0.2702, 0.4494, 0.4199, 0.6335]



'Conservation efforts are underway, but sustainable practices need to be\nadopted globally to preserve this natural treasure for future generations.'

In [ ]:
query = "How do LLMs work?"
find_answer(query, text_list, emb_list)

Similarity values between query and all sentences in the list:
[-0.03, -0.0198, 0.0232, -0.0678, -0.0234]



"The rainforest also plays a crucial role in regulating the Earth's climate\nby absorbing large amounts of carbon dioxide."

**Issues:**

1. Is the most similar text always the best answer?



2. What if the text list you have does not have the actual answer?


3. What if the answer is spread over several text chunks?


4. How to deal with PDFs and other data sources?


# Smarter RAG System

In [ ]:
url = "https://en.wikipedia.org/wiki/Amazon_rainforest"

html = fetch_wikipedia_html(url)
paragraphs = extract_paragraphs(html)

print(f"Extracted {len(paragraphs)} paragraphs.")

Extracted 75 paragraphs.


In [ ]:
paragraphs[0]

# For actual RAG pipeline, a lot of text cleaning is also required.

'The Amazon rainforest,[a] also called the Amazon jungle or Amazonia, is a moist broadleaf tropical rainforest in the Amazon biome that covers most of the Amazon basin of South America. This basin encompasses 7\xa0million\xa0km2 (2.7\xa0million\xa0sq\xa0mi),[2] of which 6\xa0million\xa0km2 (2.3\xa0million\xa0sq\xa0mi) are covered by the rainforest.[3] This region includes territory belonging to nine nations and 3,344 indigenous territories.'

In [ ]:
emb_list = create_embedding_list(paragraphs)

In [ ]:
query = "What is needed to preserve nature?"
context_list = find_answer_top_k(query, paragraphs, emb_list, k=2)
for context in context_list:
    print(context)
    print("")

# Here I have not implemented a lower cut-off threshold. Thats for you to add.

Deforestation is the conversion of forested areas to non-forested areas. The main sources of deforestation in the Amazon are human settlement and the development of the land.[68] In 2022, about 20% of the Amazon rainforest has already been deforested and a further 6% was "highly degraded".[69] Research suggests that upon reaching about 20–25% (hence 0–5% more), the tipping point to flip it into a non-forest ecosystem – degraded savannah – (in eastern, southern and central Amazonia) will be reached.[70][71][72] This process of savanisation would take decades to take full effect.[69]

Environmentalists are concerned about loss of biodiversity that will result from destruction of the forest, and also about the release of the carbon contained within the vegetation, which could accelerate global warming. Amazonian evergreen forests account for about 10% of the world's terrestrial primary productivity and 10% of the carbon stores in ecosystems[117] – of the order of 1.1 × 1011 metric tonnes 

In [ ]:
context = " ".join(context_list)

# AI is like a dumb intern.
# Improvise this prompt and make it more detailed.
prompt = f"""

INSTRUCTIONS:
- Answer from the given context only.
- Think about the query carefully.

User Query: {query}.

Context: {context}
"""

get_ai_response(prompt)

'To preserve the Amazon—and nature more broadly—what is really needed is a set of concrete, actionable steps that keep the forest intact and functional:\n\n| Key Action | Why it Matters | Practical Moves |\n|------------|----------------|-----------------|\n| **Stop large‐scale deforestation** | Once the forest area falls below ~20–25%, the land starts shifting to savannah, a process that will lock in loss of biodiversity and climate regulation. | • Enforce stricter land‑use regulations.<br>• Protect remaining forest patches through formal designations (national parks, indigenous lands).<br>• Use satellite monitoring and rapid response to illegal logging. |\n| **Restore degraded areas** | Even ‘highly degraded’ zones can rebound if the right actions are taken, preventing the tipping point. | • Reforestation and assisted natural regeneration efforts.<br>• Agroforestry schemes that combine trees with sustainable crops. |\n| **Maintain ecosystem services** | The Amazon sequesters ~1.1\u20

Think:

How can we improve the AI response?